# TowerIQ Bronze Layer Inspection

Use this notebook to inspect the Bronze Parquet tables created from raw CSV files.

Bronze is the first controlled copy of raw source data. It should preserve source records while adding ingestion metadata.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.bronze_ingestion import build_storage_path
from src.ingestion.schemas import RAW_SCHEMAS
from src.utils.config import load_config
from src.utils.spark import create_spark_session

config = load_config("configs/local.yaml")
spark_config = config["spark"]
paths = config["paths"]

spark = create_spark_session(
    app_name="TowerIQ-BronzeInspection",
    master=spark_config["master"],
    aqe_enabled=bool(spark_config["adaptive_query_execution"]),
    use_pyspark_package=bool(spark_config.get("use_pyspark_package", True)),
)
spark

## Load Bronze Tables

In [ ]:
bronze_tables = {
    table_name: spark.read.parquet(build_storage_path(paths["bronze"], "tiny", table_name))
    for table_name in RAW_SCHEMAS
}

for table_name, df in bronze_tables.items():
    df.createOrReplaceTempView(f"bronze_{table_name}")
    print(table_name, df.count())

## Q1. What row counts exist in Bronze?

In [ ]:
for table_name, df in bronze_tables.items():
    print(f"{table_name}: {df.count():,}")

## Q2. Do Bronze tables contain metadata columns?

In [ ]:
metadata_columns = ["_bronze_loaded_at", "_bronze_table", "_dataset_profile", "_source_format"]
for table_name, df in bronze_tables.items():
    missing = [column for column in metadata_columns if column not in df.columns]
    print(table_name, "missing_metadata=", missing)

## Q3. Inspect sample Bronze records

In [ ]:
bronze_tables["network_events"].select(
    "event_id", "subscriber_id", "device_id", "tower_id", "event_timestamp",
    "event_type", "network_type", "status", "_bronze_table", "_dataset_profile"
).show(10, truncate=False)

## Q4. Which event types are present in Bronze network events?

In [ ]:
spark.sql("""
SELECT event_type, COUNT(*) AS records
FROM bronze_network_events
GROUP BY event_type
ORDER BY records DESC
""").show(truncate=False)

## Q5. Which network types are present in Bronze?

In [ ]:
spark.sql("""
SELECT network_type, COUNT(*) AS records
FROM bronze_network_events
GROUP BY network_type
ORDER BY records DESC
""").show(truncate=False)

## Q6. Basic Bronze time range

In [ ]:
spark.sql("""
SELECT
  MIN(event_timestamp) AS min_event_timestamp,
  MAX(event_timestamp) AS max_event_timestamp,
  MIN(ingestion_timestamp) AS min_ingestion_timestamp,
  MAX(ingestion_timestamp) AS max_ingestion_timestamp
FROM bronze_network_events
""").show(truncate=False)

## Stop Spark

In [ ]:
spark.stop()